# Assignment 3 — Linear & Regularized Regression (Loan Amount Prediction)

Generic `assn3_experiment()` function — works on any dataset with a continuous target. Dataset used here: `Loan_Dataset.csv`, target column `Loan Amount Request (USD)`.

## Assignment 3 — `assn3_experiment()`

Generic **regression** pipeline, kept simple and step-by-step on purpose (exam-friendly — no `ColumnTransformer`/`Pipeline` magic, just plain pandas + sklearn calls you can explain line by line). Separate from `assn1_preprocess`/`assn2_experiment` because regression needs different preprocessing (standardize, one-hot encode, no stratify).

Covers every objective in the manual:
- Preprocessing: missing values -> one-hot encode categoricals -> standardize
- EDA: target distribution, feature-vs-target scatter plots
- Linear, Ridge, Lasso, Elastic Net regression
- 5-fold CV performance table (MAE, MSE, RMSE, R²) for all 4 models
- GridSearchCV and RandomizedSearchCV tuning tables for Ridge/Lasso/Elastic Net
- Test-set performance table (baseline vs tuned)
- Predicted-vs-actual plot, residual plot, train-vs-validation error bar plot, coefficient comparison plot

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def assn3_experiment(
    df,
    target_column,
    test_size=0.20,
    cv_folds=5,
    random_state=42,
    ridge_alphas=[0.01, 0.1, 1, 10, 100],
    lasso_alphas=[0.001, 0.01, 0.1, 1, 10],
    enet_alphas=[0.01, 0.1, 1, 10],
    enet_l1_ratios=[0.2, 0.5, 0.8]
):
    """
    ==================================================================
    ASSIGNMENT 3 — LINEAR & REGULARIZED REGRESSION PIPELINE
    ==================================================================
    Unlike Assignment 1/2 (classification: predicting a CATEGORY),
    this predicts a CONTINUOUS NUMBER (e.g. a loan amount). That's
    why it needs its own preprocessing (StandardScaler instead of
    MinMaxScaler, one-hot encoding for text columns, no stratify) and
    its own set of models and metrics.

    STEP 1:  Clean "fake" missing values (e.g. "?" placeholders)
    STEP 2:  Inspect the data
    STEP 3:  Handle missing values
    STEP 4:  One-hot encode categorical columns
    STEP 5:  EDA plots
    STEP 6:  Split features/target + train/test split
    STEP 7:  Standardize features
    STEP 8:  Baseline models — Linear, Ridge, Lasso, Elastic Net
    STEP 9:  5-fold Cross Validation for all 4 models
    STEP 10: Hyperparameter tuning — GridSearchCV & RandomizedSearchCV
    STEP 11: Test set performance — tuned models
    STEP 12: Predicted vs Actual + Residual plot
    STEP 13: Training vs Validation error
    STEP 14: Coefficient comparison
    """

    df = df.copy()  # work on a copy so we never modify the caller's original dataframe

    # ==============================================================
    # STEP 1: CLEAN "FAKE" MISSING VALUES
    # ==============================================================
    # Real-world datasets often mark a missing value with a placeholder
    # like "?", "NA", "-", or an empty string INSTEAD of a real blank
    # cell. Pandas has no way to know "?" secretly means "missing" —
    # it just reads the column as text, and since it looks like text,
    # df.isnull() will report ZERO missing values even though the data
    # is actually incomplete. This silently breaks two things later:
    #   1. our median/mode filling never runs on these columns
    #   2. a genuinely numeric column (e.g. a price) gets treated as
    #      categorical and one-hot encoded into thousands of columns
    # So before anything else, we swap these placeholders for real NaN.
    missing_placeholders = ["?", "NA", "N/A", "na", "n/a", "-", "None", "none", "null", "NULL", ""]
    df = df.replace(missing_placeholders, np.nan)

    # Now that the fake placeholders are gone, some columns that were
    # forced into text (object) dtype just because of a stray "?" can
    # go back to being real numeric columns. We try to convert every
    # non-target column to numeric; if a column is GENUINELY text
    # (like "Gender" -> Male/Female), the conversion fails and we just
    # leave that column alone.
    for col in df.columns:
        if col == target_column:
            continue
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            pass

    # ==============================================================
    # STEP 2: BASIC INSPECTION
    # ==============================================================
    print("=" * 70)
    print("DATASET INFORMATION")
    print("=" * 70)
    print("\nShape :", df.shape)
    display(df.head())
    df.info()

    # ==============================================================
    # STEP 3: MISSING VALUES
    # ==============================================================
    print("\nMissing Values Before Cleaning")
    print(df.isnull().sum())

    # Rows with a missing TARGET can't be used for training or testing
    # (we wouldn't know the "correct answer" to compare against), so
    # we drop them.
    if df[target_column].isnull().sum() > 0:
        df = df.dropna(subset=[target_column]).reset_index(drop=True)

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()
    if target_column in numeric_cols:
        numeric_cols.remove(target_column)

    # Numeric missing values -> median (robust to outliers, see assn1
    # for the full explanation). Categorical missing values -> mode
    # (most frequent category).
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())
    for col in categorical_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    print("\nMissing Values After Cleaning")
    print(df.isnull().sum())

    # ==============================================================
    # STEP 4: ENCODE CATEGORICAL COLUMNS
    # ==============================================================
    # ML models only understand numbers, not text categories like
    # "Male"/"Female" or "Graduate"/"Not Graduate". One-hot encoding
    # converts each category into its own 0/1 column.
    # e.g. a "Gender" column with values Male/Female becomes a single
    # column "Gender_Male" (1 if Male, 0 if Female).
    #
    # drop_first=True drops one category per column to avoid the
    # "dummy variable trap" — if you know Gender_Male=0, you already
    # know it must be Female, so keeping both columns would be
    # redundant (and can cause instability in linear models).
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    # ==============================================================
    # STEP 5: EDA (Exploratory Data Analysis)
    # ==============================================================

    # Target distribution — for regression, this shows us the SHAPE
    # of the numbers we're trying to predict. Is it roughly bell-
    # shaped (normal)? Skewed with a long tail (e.g. a few very large
    # loan amounts)? This matters because linear models often assume
    # the target is reasonably well-behaved.
    plt.figure(figsize=(7, 5))
    sns.histplot(df[target_column], kde=True)
    plt.title(f"Target Distribution: {target_column}")
    plt.show()

    # Feature vs target scatter plots — for each of the first few
    # numeric features, plot it against the target. A visible upward
    # or downward trend suggests that feature is a useful predictor;
    # a random cloud suggests it isn't (linearly, at least).
    for col in numeric_cols[:4]:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(x=df[col], y=df[target_column])
        plt.title(f"{col} vs {target_column}")
        plt.show()

    # ==============================================================
    # STEP 6: FEATURES / TARGET + TRAIN-TEST SPLIT
    # ==============================================================
    X = df.drop(columns=[target_column])
    y = df[target_column]
    feature_names = X.columns.tolist()  # save these now, before X becomes a numpy array

    # Note: no `stratify` here (unlike assn1). Stratify only makes
    # sense for CLASSIFICATION, where you're balancing discrete
    # classes. The target here is a continuous number, so a plain
    # random split is used instead.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # ==============================================================
    # STEP 7: STANDARDIZE FEATURES
    # ==============================================================
    # StandardScaler transforms each feature to have mean = 0 and
    # standard deviation = 1, using: (x - mean) / std_dev
    #
    # This is the traditional choice for linear/regularized regression
    # (as opposed to MinMaxScaler used in assn1) because Ridge/Lasso/
    # Elastic Net penalize the SIZE of coefficients — and that penalty
    # is only fair/comparable across features if every feature is on
    # the same scale to begin with.
    #
    # IMPORTANT: fit_transform() is called on X_train ONLY — the
    # scaler learns mean/std from training data alone. Then we just
    # transform() X_test using those SAME learned values. If we let
    # the scaler see the test set too, that would be "data leakage"
    # (information from the test set sneaking into training), which
    # makes our evaluation overly optimistic and dishonest.
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    print("\nTraining Shape :", X_train.shape)
    print("Testing Shape  :", X_test.shape)

    # ==============================================================
    # STEP 8: BASELINE MODELS — Linear, Ridge, Lasso, Elastic Net
    # ==============================================================
    # -- Linear Regression --
    # Fits a straight line (or hyperplane, in higher dimensions) that
    # minimizes the squared error between predictions and actual
    # values. No regularization — it will happily give large
    # coefficients if that reduces training error, which can lead to
    # OVERFITTING (memorizing noise in the training data instead of
    # learning the real pattern).
    #
    # -- Ridge Regression (L2 regularization) --
    # Same as Linear Regression, but adds a penalty proportional to
    # the SQUARE of the coefficients. This shrinks coefficients
    # toward zero (but rarely exactly to zero), reducing overfitting
    # and making the model less sensitive to noisy features.
    #
    # -- Lasso Regression (L1 regularization) --
    # Adds a penalty proportional to the ABSOLUTE VALUE of the
    # coefficients. Unlike Ridge, this can shrink some coefficients
    # all the way to EXACTLY zero — effectively removing those
    # features from the model. This makes Lasso useful for automatic
    # FEATURE SELECTION.
    #
    # -- Elastic Net --
    # A blend of Ridge and Lasso — it has both an L1 and an L2 penalty
    # term, controlled by a mixing ratio (l1_ratio). Gets some of
    # Lasso's feature-selection behaviour and some of Ridge's
    # stability, useful when you have many correlated features.
    models = {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(),
        "Lasso Regression": Lasso(),
        "Elastic Net Regression": ElasticNet()
    }

    def get_metrics(y_true, y_pred):
        # MAE  (Mean Absolute Error)     -> average of |actual - predicted|.
        #      Easy to interpret: "on average, we're off by this much".
        # MSE  (Mean Squared Error)      -> average of (actual - predicted)^2.
        #      Squaring punishes LARGE errors much more than small ones.
        # RMSE (Root Mean Squared Error) -> square root of MSE. Brings the
        #      units back to the same scale as the target (e.g. rupees,
        #      not rupees-squared), while still penalizing big misses.
        # R2   (R-squared)               -> how much of the variance in the
        #      target our model explains, from 0 to 1 (can go negative for
        #      a model worse than just predicting the average every time).
        #      1.0 = perfect fit, 0 = no better than predicting the mean.
        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = mse ** 0.5
        r2 = r2_score(y_true, y_pred)
        return mae, mse, rmse, r2

    test_rows = []
    fitted_models = {}

    for name, model in models.items():
        start = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start
        fitted_models[name] = model

        pred = model.predict(X_test)
        mae, mse, rmse, r2 = get_metrics(y_test, pred)

        test_rows.append({
            "Model": name, "MAE": mae, "MSE": mse,
            "RMSE": rmse, "R2": r2, "Train Time": train_time
        })

    test_table = pd.DataFrame(test_rows)
    print("\nTest Set Performance (baseline models)")
    display(test_table)

    # ==============================================================
    # STEP 9: 5-FOLD CROSS VALIDATION for all 4 models
    # ==============================================================
    # Same idea as assn2's cross-validation: split the training data
    # into 5 folds, train on 4, validate on 1, repeat 5 times with a
    # different fold held out each time, then average the results.
    # This gives a more trustworthy performance estimate than a
    # single split, and is what the lab manual specifically asks for.
    cv_rows = []
    for name, model in models.items():
        # scoring="r2" -> cross_val_score returns an R2 score per fold
        r2_scores = cross_val_score(model, X_train, y_train, cv=cv_folds, scoring="r2")

        # scikit-learn reports errors as NEGATIVE numbers internally
        # (so that "higher score = better" stays consistent across all
        # metrics) — we negate them back to get normal positive MAE/MSE.
        mae_scores = -cross_val_score(model, X_train, y_train, cv=cv_folds, scoring="neg_mean_absolute_error")
        mse_scores = -cross_val_score(model, X_train, y_train, cv=cv_folds, scoring="neg_mean_squared_error")

        cv_rows.append({
            "Model": name,
            "MAE": mae_scores.mean(),
            "MSE": mse_scores.mean(),
            "RMSE": mse_scores.mean() ** 0.5,
            "R2": r2_scores.mean()
        })

    cv_table = pd.DataFrame(cv_rows)
    print(f"\n{cv_folds}-Fold Cross Validation Performance")
    display(cv_table)

    # ==============================================================
    # STEP 10: HYPERPARAMETER TUNING — Grid Search + Randomized Search
    # ==============================================================
    # Ridge, Lasso, and Elastic Net all have an "alpha" hyperparameter
    # that controls HOW STRONG the regularization penalty is:
    #   alpha = 0     -> no regularization at all (same as plain Linear Regression)
    #   small alpha   -> weak penalty, coefficients can stay large
    #   large alpha   -> strong penalty, coefficients shrink a lot
    #                    (too large -> UNDERFITTING, model becomes too simple)
    # Elastic Net additionally has "l1_ratio" (0 to 1), which controls
    # the BLEND between Ridge-style (l1_ratio=0) and Lasso-style
    # (l1_ratio=1) penalties.
    #
    # We don't know the best alpha/l1_ratio in advance, so we search
    # for it using cross-validation — trying each candidate value and
    # picking whichever gives the best average CV score.
    search_space = {
        "Ridge Regression": (Ridge(), {"alpha": ridge_alphas}),
        "Lasso Regression": (Lasso(), {"alpha": lasso_alphas}),
        "Elastic Net Regression": (ElasticNet(), {"alpha": enet_alphas, "l1_ratio": enet_l1_ratios})
    }

    tuning_rows = []
    # Linear Regression has no hyperparameters to tune, so we just
    # reuse the already-fitted baseline model for it.
    tuned_models = {"Linear Regression": fitted_models["Linear Regression"]}

    for name, (estimator, grid) in search_space.items():

        # GridSearchCV -> exhaustively tries every value/combination
        # in the grid, cross-validating each one, and keeps the best.
        grid_search = GridSearchCV(estimator, grid, cv=cv_folds, scoring="r2")
        grid_search.fit(X_train, y_train)

        tuning_rows.append({
            "Model": name, "Search Method": "GridSearchCV",
            "Best Params": grid_search.best_params_, "Best CV R2": grid_search.best_score_
        })

        # RandomizedSearchCV -> randomly samples a fixed number
        # (n_iter=8) of combinations instead of trying all of them.
        # Faster, especially useful when the grid has many combinations
        # (like Elastic Net's alpha x l1_ratio grid).
        random_search = RandomizedSearchCV(
            estimator, grid, cv=cv_folds, scoring="r2",
            n_iter=8, random_state=random_state
        )
        random_search.fit(X_train, y_train)

        tuning_rows.append({
            "Model": name, "Search Method": "RandomizedSearchCV",
            "Best Params": random_search.best_params_, "Best CV R2": random_search.best_score_
        })

        # We keep GridSearchCV's winning model to carry forward into
        # the next steps (it's exhaustive, so it's at least as good as
        # RandomizedSearchCV's result on this grid).
        tuned_models[name] = grid_search.best_estimator_

    tuning_table = pd.DataFrame(tuning_rows)
    print("\nHyperparameter Tuning Summary (Grid vs Randomized Search)")
    display(tuning_table)

    # ==============================================================
    # STEP 11: TEST SET PERFORMANCE — tuned models
    # ==============================================================
    # Now that each model has its best hyperparameters, retrain it
    # (freshly) and check its performance on the untouched test set —
    # this is the final, honest measure of how good each tuned model
    # really is.
    test_rows_tuned = []
    for name, model in tuned_models.items():
        start = time.time()
        model.fit(X_train, y_train)
        train_time = time.time() - start
        pred = model.predict(X_test)
        mae, mse, rmse, r2 = get_metrics(y_test, pred)
        test_rows_tuned.append({
            "Model": name, "MAE": mae, "MSE": mse,
            "RMSE": rmse, "R2": r2, "Train Time": train_time
        })

    test_table_tuned = pd.DataFrame(test_rows_tuned)
    print("\nTest Set Performance (tuned models)")
    display(test_table_tuned)

    # ==============================================================
    # STEP 12: PREDICTED VS ACTUAL + RESIDUAL PLOT (best model by R2)
    # ==============================================================
    # Pick whichever tuned model scored the highest R2 on the test set.
    best_row = test_table_tuned.sort_values("R2", ascending=False).iloc[0]
    best_name = best_row["Model"]
    best_model = tuned_models[best_name]
    best_pred = best_model.predict(X_test)

    # Predicted vs Actual plot: if the model were PERFECT, every point
    # would fall exactly on the diagonal red dashed line (predicted ==
    # actual). The more scattered the points are around that line, the
    # worse the model's predictions.
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, best_pred, alpha=0.5)
    lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
    plt.plot(lims, lims, "r--")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"Predicted vs Actual ({best_name})")
    plt.show()

    # Residuals = actual - predicted, i.e. how wrong each prediction
    # was (and in which direction). A GOOD model's residuals should
    # look like a random, formless cloud scattered evenly around 0 —
    # that means the errors are just noise. If you instead see a
    # clear PATTERN (a curve, a funnel shape, a slope), it means the
    # model is systematically missing something in the data.
    residuals = y_test - best_pred
    plt.figure(figsize=(7, 5))
    plt.scatter(best_pred, residuals, alpha=0.5)
    plt.axhline(0, color="r", linestyle="--")
    plt.xlabel("Predicted")
    plt.ylabel("Residual")
    plt.title(f"Residual Plot ({best_name})")
    plt.show()

    # ==============================================================
    # STEP 13: TRAINING vs VALIDATION ERROR
    # ==============================================================
    # For each baseline model, compute its error on the TRAINING data
    # it learned from, and separately its error on the TEST data it
    # has never seen. Comparing the two tells us about overfitting:
    #   - Training error LOW, Test error HIGH  -> OVERFITTING
    #     (model memorized training data, doesn't generalize)
    #   - Both errors HIGH                      -> UNDERFITTING
    #     (model is too simple to capture the real pattern)
    #   - Both errors LOW and close together    -> good fit
    train_errors, val_errors = [], []
    for name, model in models.items():
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        train_errors.append(mean_squared_error(y_train, train_pred))
        val_errors.append(mean_squared_error(y_test, test_pred))

    x = np.arange(len(models))
    plt.figure(figsize=(8, 5))
    plt.bar(x - 0.2, train_errors, width=0.4, label="Training Error (MSE)")
    plt.bar(x + 0.2, val_errors, width=0.4, label="Validation Error (MSE)")
    plt.xticks(x, list(models.keys()), rotation=15)
    plt.ylabel("MSE")
    plt.title("Training vs Validation Error")
    plt.legend()
    plt.show()

    # ==============================================================
    # STEP 14: COEFFICIENT COMPARISON
    # ==============================================================
    # Each linear-type model assigns a "coefficient" (weight) to every
    # feature, showing how much that feature pushes the prediction up
    # or down. Comparing coefficients ACROSS models shows regularization
    # in action:
    #   - Ridge's coefficients should generally be SMALLER in magnitude
    #     than plain Linear Regression's.
    #   - Lasso's coefficients will often include EXACT ZEROS (features
    #     it decided to drop entirely).
    #   - Elastic Net usually lands somewhere between the two.
    coef_table = pd.DataFrame(
        {name: model.coef_ for name, model in tuned_models.items()},
        index=feature_names
    )

    # Only plot the 10 features with the largest total coefficient
    # magnitude (across all models), otherwise the chart becomes
    # unreadable if there are many features.
    top_features = coef_table.abs().sum(axis=1).sort_values(ascending=False).head(10).index
    coef_table.loc[top_features].plot(kind="bar", figsize=(12, 6))
    plt.title("Coefficient Comparison (Top 10 Features)")
    plt.ylabel("Coefficient Value")
    plt.tight_layout()
    plt.show()

    return {
        "cv_table": cv_table,
        "test_table": test_table,
        "tuning_table": tuning_table,
        "test_table_tuned": test_table_tuned,
        "coef_table": coef_table,
        "fitted_models": fitted_models,
        "tuned_models": tuned_models,
        "best_model_name": best_name,
        "scaler": scaler,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }

### Usage — Assignment 3

Swap `"loan_data.csv"` and `"LoanAmount"` for any other regression dataset/target and it works the same way.

In [ ]:
df3 = pd.read_csv("Loan_Dataset.csv")

# Customer ID / Name / Property ID are identifiers, not real predictive
# features - drop them so one-hot encoding doesn't explode into
# thousands of useless columns.
df3 = df3.drop(columns=["Customer ID", "Name", "Property ID"])

exp3_output = assn3_experiment(df3, "Loan Amount Request (USD)")

exp3_output["test_table_tuned"].to_csv("Experiment3_Results.csv", index=False)
exp3_output["test_table_tuned"]